# 🚨 Notebook 4: Debugging an Outage End-to-End

This is the capstone. We'll run a tiny simulated **checkout service** that emits all three pillars (logs, metrics, traces) with a shared `trace_id`. Then we'll inject a problem, "get paged" by a metric, and walk the full investigation:

1. **Metrics → "what?"** A dashboard alert fires.
2. **Traces → "where?"** Find a slow trace; see which step is dragging.
3. **Logs → "why?"** Pull the log lines for that exact `trace_id` to see the error.

By the end you should see **why all three pillars matter** — and how they snap together when one ID flows through everything.

## Learning objectives
- Build a small instrumented service end-to-end.
- Practice the **alert → trace → log** investigation loop.
- See first-hand why the **shared trace_id** is the glue.

## 🧰 The instrumentation library

A minimal version of what an OpenTelemetry SDK would give you: a metrics registry, a structured logger, and a span tree. They all share `trace_id`.

In [ ]:
import time, json, uuid, random
from collections import defaultdict
from contextlib import contextmanager
from contextvars import ContextVar

# A context variable so deeply-nested code can find the active trace_id
# without having to pass it as an argument everywhere.
_current_trace: ContextVar[str | None] = ContextVar("trace_id", default=None)

LOG_LINES: list[dict] = []  # capture logs in-memory so we can search them later

def log(level, event, **fields):
    record = {
        "ts": time.strftime("%H:%M:%S"),
        "level": level,
        "event": event,
        "trace_id": _current_trace.get(),
        **fields,
    }
    LOG_LINES.append(record)
    # In real life this prints to stdout / ships to Loki / Elasticsearch.

class Metrics:
    def __init__(self):
        self.counters = defaultdict(float)
        self.samples = defaultdict(list)
    def inc(self, name, by=1, **labels):
        self.counters[(name, tuple(sorted(labels.items())))] += by
    def observe(self, name, v, **labels):
        self.samples[(name, tuple(sorted(labels.items())))].append(v)
    def p(self, name, q, **labels):
        s = sorted(self.samples.get((name, tuple(sorted(labels.items()))), []))
        if not s: return None
        return s[int(q * (len(s) - 1))]

M = Metrics()

class Span:
    def __init__(self, name, parent=None):
        self.name = name
        self.parent = parent
        self.trace_id = parent.trace_id if parent else uuid.uuid4().hex[:8]
        self.duration_ms = 0.0
        self.children = []
        if parent:
            parent.children.append(self)
    def __enter__(self):
        self._t0 = time.perf_counter()
        self._token = _current_trace.set(self.trace_id)
        return self
    def __exit__(self, *a):
        self.duration_ms = self.elapsed_ms()
        _current_trace.reset(self._token)
    def elapsed_ms(self):
        """Time since the span opened. `duration_ms` is only final after __exit__,
        so anything logging from *inside* the block must use this instead."""
        return (time.perf_counter() - self._t0) * 1000
    def show(self, indent=0):
        bar = "└─" if indent else "•"
        print(f"{' ' * indent}{bar} {self.name:<25} {self.duration_ms:7.1f} ms  trace={self.trace_id}")
        for c in self.children:
            c.show(indent + 4)


## 🛍️ The simulated service

A `checkout()` function that calls auth → db → payments → email. We instrument every step. The `payments_charge` step is where we'll inject our problem later.

In [ ]:
# Toggle these to inject problems.
PAYMENTS_SLOW_PROBABILITY = 0.0   # fraction of requests where payments hangs
PAYMENTS_FAIL_PROBABILITY = 0.0   # fraction of requests where payments errors

def checkout(user):
    with Span("GET /checkout") as root:
        log("INFO", "request.start", user=user, path="/checkout")
        M.inc("http_requests_total", endpoint="/checkout")
        try:
            with Span("auth.verify_jwt", parent=root):
                time.sleep(random.uniform(0.001, 0.003))

            with Span("db.read_cart", parent=root):
                time.sleep(random.uniform(0.005, 0.015))

            with Span("payments.charge", parent=root):
                # Inject latency / failure here:
                if random.random() < PAYMENTS_SLOW_PROBABILITY:
                    time.sleep(random.uniform(0.4, 0.8))   # SLOW path
                    log("WARN", "payments.slow", provider="stripe")
                else:
                    time.sleep(random.uniform(0.03, 0.08))
                if random.random() < PAYMENTS_FAIL_PROBABILITY:
                    raise RuntimeError("card_declined")

            with Span("email.send_receipt", parent=root):
                time.sleep(random.uniform(0.002, 0.005))

            # NOTE: root.duration_ms is still 0 here — the span has not exited yet.
            # Logging it directly would put "ms": 0.0 on every single line.
            log("INFO", "request.end", status=200, ms=round(root.elapsed_ms(), 1))
            M.inc("http_requests_total", endpoint="/checkout", status="200")
        except Exception as e:
            log("ERROR", "request.failed", error=str(e), ms=round(root.elapsed_ms(), 1))
            M.inc("http_requests_total", endpoint="/checkout", status="500")
        return root

USERS = ["alice", "bob", "carol", "dan", "eve"]

## 1️⃣ Healthy baseline

Run 200 healthy requests. Compute the SLI ("% successful in < 200 ms") and the p95 latency.

In [ ]:
random.seed(1)
PAYMENTS_SLOW_PROBABILITY = 0.0
PAYMENTS_FAIL_PROBABILITY = 0.0

LOG_LINES.clear()
M.counters.clear(); M.samples.clear()

baseline_traces = []
for _ in range(200):
    root = checkout(random.choice(USERS))
    M.observe("http_latency_ms", root.duration_ms, endpoint="/checkout")
    baseline_traces.append(root)

successes = M.counters.get(("http_requests_total", (("endpoint","/checkout"),("status","200"))), 0)
total     = M.counters.get(("http_requests_total", (("endpoint","/checkout"),)), 0)
p50 = M.p("http_latency_ms", 0.50, endpoint="/checkout")
p95 = M.p("http_latency_ms", 0.95, endpoint="/checkout")
p99 = M.p("http_latency_ms", 0.99, endpoint="/checkout")
print(f"requests: {int(total)}   successes: {int(successes)}   SLI: {successes/total:.2%}")
print(f"latency  p50={p50:.1f} ms  p95={p95:.1f} ms  p99={p99:.1f} ms")

baseline_p50, baseline_p95 = p50, p95
assert successes == total, "the healthy baseline should have no failures"
assert p50 <= p95 <= p99
assert p95 < 200, f"baseline p95 {p95:.0f} ms should be inside the 200 ms SLO"

# The request.end log line must carry a real duration, not a zero.
end_lines = [r for r in LOG_LINES if r["event"] == "request.end"]
assert all(r["ms"] > 0 for r in end_lines), "duration_ms is only final after the span exits"

## 2️⃣ Inject a problem and "get paged"

A bad deploy at the payments provider has made 30% of charges very slow. Run another 200 requests.

In [ ]:
random.seed(2)
PAYMENTS_SLOW_PROBABILITY = 0.30
PAYMENTS_FAIL_PROBABILITY = 0.0

LOG_LINES.clear()
M.counters.clear(); M.samples.clear()

degraded_traces = []
for _ in range(200):
    root = checkout(random.choice(USERS))
    M.observe("http_latency_ms", root.duration_ms, endpoint="/checkout")
    degraded_traces.append(root)

p50 = M.p("http_latency_ms", 0.50, endpoint="/checkout")
p95 = M.p("http_latency_ms", 0.95, endpoint="/checkout")
p99 = M.p("http_latency_ms", 0.99, endpoint="/checkout")
print(f"📟 ALERT: p95 latency = {p95:.0f} ms (SLO threshold = 200 ms)")
print(f"latency  p50={p50:.1f} ms  p95={p95:.1f} ms  p99={p99:.1f} ms")

# The alert has to be justified by the data, not by the print statement.
assert p95 > 200, "p95 should have breached the 200 ms SLO"
assert p95 > 3 * baseline_p95, (p95, baseline_p95)
# ...while the median barely moves: 70% of requests took the fast path and those
# users notice nothing. That asymmetry is exactly why you alert on p95/p99.
print(f"p50: {baseline_p50:.0f} -> {p50:.0f} ms ({p50/baseline_p50:.1f}x)   "
      f"p95: {baseline_p95:.0f} -> {p95:.0f} ms ({p95/baseline_p95:.1f}x)")
assert p50 < 2 * baseline_p50, "the median should be almost unaffected"

**Step 1 done.** Metrics told us *what*: p95 is way over our SLO. Metrics can't tell us *why* — they're just numbers.

## 3️⃣ Drill into traces — *where* is the time going?

Find the slowest traces and look at their span trees.

In [ ]:
slow_traces = sorted(degraded_traces, key=lambda t: -t.duration_ms)[:3]
for i, t in enumerate(slow_traces, 1):
    print(f"--- Slow trace #{i} ({t.duration_ms:.0f} ms) ---")
    t.show()
    print()

# In each slow trace the dominant child span must be payments.charge — that is the
# claim the investigation rests on, so check it instead of squinting at the tree.
for t in slow_traces:
    worst = max(t.children, key=lambda c: c.duration_ms)
    assert worst.name == "payments.charge", worst.name
    assert worst.duration_ms > 0.7 * t.duration_ms, "the culprit should dominate"
print("In all 3 slow traces, payments.charge is >70% of the total time ✔")

**Step 2 done.** Traces told us *where*: the time is in `payments.charge`, not in our DB or auth code.

## 4️⃣ Pull the logs for one slow trace — *why* did it happen?

Take the slowest trace's `trace_id`, and filter logs by it. This is the magic of correlation: one ID, three tools.

In [ ]:
target_trace_id = slow_traces[0].trace_id
print(f"🔍 Searching logs for trace_id = {target_trace_id}\n")
matching = [r for r in LOG_LINES if r["trace_id"] == target_trace_id]
for r in matching:
    print(json.dumps(r))

# Correlation must be exact: every line we pulled belongs to this request, and the
# smoking gun (payments.slow) is among them.
assert matching, "no logs found for the slowest trace — correlation is broken"
assert all(r["trace_id"] == target_trace_id for r in matching)
assert any(r["event"] == "payments.slow" for r in matching)

# And the filter is doing real work: this is a handful of lines out of hundreds.
print(f"\n{len(matching)} lines matched out of {len(LOG_LINES)} captured "
      f"({len(matching)/len(LOG_LINES):.1%}) — that is the value of one shared id.")

**Step 3 done.** Logs told us *why*: a `payments.slow` warning was emitted by the Stripe provider for that exact request.

## 🧠 What just happened

| Step | Pillar | Question | What we learned |
|---|---|---|---|
| 1 | Metrics | *What?* | p95 spiked above SLO |
| 2 | Traces | *Where?* | `payments.charge` is dragging |
| 3 | Logs | *Why?* | Stripe provider emitted slow warnings |

Without **all three** you'd be guessing:

- Metrics alone → "checkout is slow" — but you don't know which step.
- Traces alone → no proactive alert; you'd discover it from a customer tweet.
- Logs alone → drowning in millions of unstructured lines, no `trace_id` to filter by.

The shared `trace_id` is the glue.

## 🧪 Try it yourself

Tweak the simulation and re-run:

1. Set `PAYMENTS_FAIL_PROBABILITY = 0.05` and watch the **error rate** SLI fall.
2. Move the `time.sleep` from `payments.charge` into `db.read_cart` — does your investigation still find the right culprit?
3. Add a new span (e.g. `inventory.check`) inside `checkout()` and verify it shows up in the trace tree.

## ✅ You've now seen the whole observability story

- The **three pillars** and why each is bad at the others' job (NB1).
- How to set **SLOs** and **alert on burn rate** so on-call isn't a disaster (NB2).
- How a real metrics collector works under the hood, including the **cardinality trap** (NB3).
- How an **alert → trace → log** investigation actually plays out (this notebook).

That's enough observability to design a system, set up a dashboard, and survive a 3 AM page.